# Pipeline: FASTA, colecciones `tes` y `secuencias`

**Responsable:** Juan Sebastián Arias Rueda (revisión: Thomas Alejandro Toro Herrera)

**Objetivo:** Leer y validar el FASTA, cruzar con `especies`, calcular atributos y cargar `tes` y `secuencias` por lotes.

Modelo de referencia: `../MODELO.md`

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["panteon"]

In [2]:
import random
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from Bio import SeqIO
from bson import ObjectId
from pymongo import ASCENDING

sys.path.insert(0, "..")
from src.benchmark import guardar_benchmark
from src.normalizacion import normalize_species

FASTA = Path("../data/PanTEon_Database_v1.6.1.fasta")
RESULTADOS = Path("../resultados")
LOTE = 2000
DB_NAME = db.name
# True solo si después se ejecuta 03_pipeline_kmers.ipynb (cadena completa 02 → 03)
RECONSTRUIR_TODO = False

# Este notebook borra tes y secuencias; kmers y {DB_NAME}_bench.tes_embebidos dependen de los _id de tes
dependientes = {
    f"{DB_NAME}.kmers": db.kmers.estimated_document_count(),
    f"{DB_NAME}_bench.tes_embebidos": client[f"{DB_NAME}_bench"].tes_embebidos.estimated_document_count(),
}
con_datos = {nombre: n for nombre, n in dependientes.items() if n}
if con_datos and not RECONSTRUIR_TODO:
    raise RuntimeError(
        f"Re-ejecutar este notebook cambia los _id de tes y deja huérfanas colecciones con datos: {con_datos}. "
        "Poner RECONSTRUIR_TODO = True y ejecutar después 03_pipeline_kmers.ipynb, o restaurar el dump "
        "(resultados/dump_info.md)."
    )

## 1. Lectura y validación del FASTA

Primera pasada: se recorre el FASTA con Biopython y se guarda solo lo medido de cada registro (sin las secuencias), para validar todo antes de cargar. El encabezado se separa por `#` y `@` sin asumir el prefijo `PDB` (31.600 IDs no lo tienen; ver `resultados/inconsistencias_fasta.csv`).

In [3]:
def parsear_encabezado(header):
    """'ID#CLASE/ORDEN/SUPERFAMILIA @Especie' -> dict, o None si está mal formado."""
    if header.count("#") != 1 or header.count("@") != 1:
        return None
    izq, especie = header.split("@")
    pdb_id, clasificacion = izq.split("#")
    niveles = clasificacion.strip().split("/")
    return {
        "pdb_id": pdb_id.strip(),
        "niveles": niveles,
        "species_name": especie.strip(),
    }


def atributos(seq):
    """length, gc_content y n_percent sobre la secuencia en mayúsculas."""
    up = seq.upper()
    a, c, g, t, n = (up.count(b) for b in "ACGTN")
    validas = a + c + g + t
    return {
        "length": len(seq),
        "gc_content": (g + c) / validas if validas else None,
        "n_percent": n / len(seq) if seq else None,
    }, set(up)


t0 = time.perf_counter()
registros, mal_formados, pocos_niveles = [], [], []
caracteres = Counter()
for rec in SeqIO.parse(FASTA, "fasta"):
    enc = parsear_encabezado(rec.description)
    if enc is None:
        mal_formados.append(rec.description)
        continue
    if len(enc["niveles"]) < 3 or not all(enc["niveles"]):
        pocos_niveles.append(rec.description)
    attrs, chars = atributos(str(rec.seq))
    caracteres.update(chars)
    registros.append({**enc, **attrs})
t_primera_pasada = time.perf_counter() - t0

meta = pd.DataFrame(registros)
print(f"Registros: {len(meta)} | primera pasada: {t_primera_pasada:.1f} s")
print("Encabezados mal formados:", len(mal_formados), mal_formados[:5])
print("Clasificaciones con menos de 3 niveles:", len(pocos_niveles), pocos_niveles[:5])

Registros: 239946 | primera pasada: 9.2 s
Encabezados mal formados: 0 []
Clasificaciones con menos de 3 niveles: 0 []


In [4]:
duplicados = meta["pdb_id"][meta["pdb_id"].duplicated(keep=False)]
print("pdb_id duplicados:", duplicados.nunique())
assert duplicados.empty, f"pdb_id duplicados: {sorted(duplicados.unique())[:20]}. Detener y revisar."
assert not mal_formados and not pocos_niveles, "Encabezados a revisar antes de cargar"

otros = sorted(set(caracteres) - set("ACGTN"))
print("Caracteres (en mayúsculas) fuera de A, C, G, T, N:", otros)
print("Longitud mínima:", meta["length"].min(), "| máxima:", meta["length"].max(),
      "| longitud cero:", (meta["length"] == 0).sum())
print("Sin ninguna A, C, G o T (gc_content indefinido):", meta["gc_content"].isna().sum())
print("Prefijo distinto de PDB + 8 dígitos:", (~meta["pdb_id"].str.fullmatch(r"PDB\d{8}")).sum())

pdb_id duplicados: 0
Caracteres (en mayúsculas) fuera de A, C, G, T, N: []
Longitud mínima: 81 | máxima: 19926 | longitud cero: 0
Sin ninguna A, C, G o T (gc_content indefinido): 0
Prefijo distinto de PDB + 8 dígitos: 31600


## 2. Cruce con `especies` y reporte de lo que no cruza

In [5]:
especies = {normalize_species(e["species"]): e for e in db.especies.find(
    {}, {"species": 1, "taxonomy.kingdom": 1, "taxonomy.phylum": 1, "source": 1})}
species_id = {llave: e["_id"] for llave, e in especies.items()}

meta["llave"] = meta["species_name"].map(normalize_species)
sin_cruce = meta[~meta["llave"].isin(species_id)]
print("Especies en el diccionario:", len(species_id))
print("TEs que no cruzan:", len(sin_cruce), sin_cruce["species_name"].value_counts().head(10).to_dict())
assert sin_cruce.empty, "Hay TEs sin especie: detener y revisar (se esperan 0)"

Especies en el diccionario: 2790
TEs que no cruzan: 0 {}


## 3. Atributos: longitud, GC, %N

**Decisión:** `gc_content = (G + C) / (A + C + G + T)`: el denominador excluye N y códigos IUPAC, porque una posición ambigua no se sabe si es G/C o A/T; así una secuencia con muchas N no baja su GC artificialmente. `n_percent = N / length` usa la longitud total. Ambos se calculan sobre la secuencia en mayúsculas (el FASTA tiene minúsculas de enmascaramiento), pero en `secuencias` se guarda la secuencia original sin modificar.

In [6]:
meta[["length", "gc_content", "n_percent"]].describe().round(4)

,length,gc_content,n_percent
count,239946.0000,239946.0000,239946.0000
mean,4017.1598,0.4319,0.0038
std,2751.3267,0.0762,0.0119
min,81.0000,0.1121,0.0000
25%,1463.0000,0.3813,0.0000
50%,4259.0000,0.4230,0.0000
75%,5656.0000,0.4775,0.0000
max,19926.0000,0.8039,0.5006


## 4. Validación `$jsonSchema` de `tes`

In [7]:
def bson_tipo(*tipos):
    return {"bsonType": list(tipos) if len(tipos) > 1 else tipos[0]}


ESQUEMA_TES = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["pdb_id", "species_id", "species_name", "kingdom", "phylum", "source",
                     "classification", "length", "gc_content", "n_percent"],
        "properties": {
            "pdb_id": bson_tipo("string"),
            "species_id": bson_tipo("objectId"),
            "species_name": bson_tipo("string"),
            "kingdom": {"enum": ["Animalia", "Plantae", "Fungi"]},
            # phylum puede faltar en especies (3 casos en el CSV): se copia null
            "phylum": bson_tipo("string", "null"),
            "source": bson_tipo("string"),
            "classification": {
                "bsonType": "object",
                "required": ["te_class", "te_order", "te_superfamily"],
                "properties": {k: bson_tipo("string") for k in ["te_class", "te_order", "te_superfamily"]},
            },
            "length": {"bsonType": ["int", "long"], "minimum": 1},
            "gc_content": {"bsonType": "double", "minimum": 0, "maximum": 1},
            "n_percent": {"bsonType": "double", "minimum": 0, "maximum": 1},
        },
    }
}

# Borra y recrea: el notebook se puede ejecutar varias veces con el mismo resultado
for nombre in ["tes", "secuencias"]:
    db.drop_collection(nombre)
db.create_collection("tes", validator=ESQUEMA_TES, validationLevel="strict", validationAction="error")
db.create_collection("secuencias")

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'panteon'), 'secuencias')

## 5. Carga por lotes (`insert_many`)

Segunda pasada sobre el FASTA. El `ObjectId` se genera en Python y se usa en el documento de `tes` y en el de `secuencias`. Se mide aparte el armado de documentos (lectura + atributos) y la inserción.

In [8]:
def documentos(rec, m):
    _id = ObjectId()
    esp = especies[m["llave"]]
    te_class, te_order, te_superfamily = m["niveles"]
    te = {
        "_id": _id,
        "pdb_id": m["pdb_id"],
        "species_id": esp["_id"],
        "species_name": m["species_name"],
        "kingdom": esp["taxonomy"]["kingdom"],
        "phylum": esp["taxonomy"].get("phylum"),
        "source": esp["source"],
        "classification": {"te_class": te_class, "te_order": te_order, "te_superfamily": te_superfamily},
        "length": int(m["length"]),
        "gc_content": float(m["gc_content"]),
        "n_percent": float(m["n_percent"]),
    }
    return te, {"_id": _id, "sequence": str(rec.seq)}


def insertar(tes, secs):
    t0 = time.perf_counter()
    db.tes.insert_many(tes, ordered=False)
    db.secuencias.insert_many(secs, ordered=False)
    return time.perf_counter() - t0


filas_meta = meta.to_dict("records")
t_calculo = t_insercion = 0.0
tes_lote, secs_lote = [], []
t0 = time.perf_counter()
for rec, m in zip(SeqIO.parse(FASTA, "fasta"), filas_meta):
    assert parsear_encabezado(rec.description)["pdb_id"] == m["pdb_id"]
    te, sec = documentos(rec, m)
    tes_lote.append(te)
    secs_lote.append(sec)
    if len(tes_lote) == LOTE:
        t_calculo += time.perf_counter() - t0
        t_insercion += insertar(tes_lote, secs_lote)
        tes_lote, secs_lote = [], []
        t0 = time.perf_counter()
t_calculo += time.perf_counter() - t0
if tes_lote:
    t_insercion += insertar(tes_lote, secs_lote)

print(f"Armado: {t_calculo:.1f} s | inserción: {t_insercion:.1f} s")

Armado: 2.1 s | inserción: 3.9 s


## 6. Índices

In [9]:
t0 = time.perf_counter()
db.tes.create_index([("pdb_id", ASCENDING)], unique=True)
db.tes.create_index([("species_id", ASCENDING)])
db.tes.create_index([("species_name", ASCENDING), ("classification.te_superfamily", ASCENDING)])
db.tes.create_index([("kingdom", ASCENDING)])
t_indices = time.perf_counter() - t0

n_tes = db.tes.count_documents({})
benchmark = guardar_benchmark([{
    "estrategia": "tes_y_secuencias", "base": DB_NAME, "n_tes": n_tes,
    "n_docs": n_tes + db.secuencias.count_documents({}),
    "t_calculo_s": round(t_calculo, 3), "t_insercion_s": round(t_insercion, 3), "t_indices_s": round(t_indices, 3),
}], RESULTADOS / "benchmark_carga.csv")
print(sorted(db.tes.index_information()))
benchmark

['_id_', 'kingdom_1', 'pdb_id_1', 'species_id_1', 'species_name_1_classification.te_superfamily_1']


,estrategia,base,n_tes,n_docs,t_calculo_s,t_insercion_s,t_indices_s
0,coleccion_kmers,panteon_prueba,25,150,0.005,0.004,0.086
1,embebido_en_tes,panteon_prueba_bench,25,25,0.005,0.006,0.036
2,tes_y_secuencias,panteon,239946,479892,2.131,3.875,0.559


## 7. Verificación

In [10]:
ESPERADO = 239946
assert db.tes.count_documents({}) == ESPERADO == db.secuencias.count_documents({})

ids_tes = db.tes.distinct("_id")
ids_secs = db.secuencias.distinct("_id")
assert set(ids_tes) == set(ids_secs), "Los _id de tes y secuencias no coinciden"

ids_especies = set(db.especies.distinct("_id"))
assert set(db.tes.distinct("species_id")) <= ids_especies, "Hay species_id que no existen en especies"
print(f"{ESPERADO} documentos en cada colección, _id idénticos y todo species_id existe en especies")

# 5 TEs al azar reconstruyen su encabezado original exacto
muestra = {te["pdb_id"]: te for te in db.tes.aggregate([{"$sample": {"size": 5}}])}
originales = {}
for rec in SeqIO.parse(FASTA, "fasta"):
    pdb = rec.description.split("#", 1)[0]
    if pdb in muestra:
        originales[pdb] = (rec.description, str(rec.seq))
for pdb, te in muestra.items():
    c = te["classification"]
    reconstruido = f"{te['pdb_id']}#{c['te_class']}/{c['te_order']}/{c['te_superfamily']} @{te['species_name']}"
    header, seq = originales[pdb]
    assert reconstruido == header, (reconstruido, header)
    assert db.secuencias.find_one({"_id": te["_id"]})["sequence"] == seq
    print("OK", reconstruido)

239946 documentos en cada colección, _id idénticos y todo species_id existe en especies


OK PDB00116259#CLASSI/LTR/COPIA @Echium plantagineum
OK PDB00078294#CLASSI/LTR/COPIA @Nicotiana attenuata
OK PDB00025062#CLASSI/LTR/GYPSY @Coregonus clupeaformis
OK PDB00133459#CLASSI/LINE/L1 @Jacaranda mimosifolia
OK PDB00174652#CLASSI/LTR/GYPSY @Themeda triandra


## Reporte de calidad

In [11]:
filas = [
    ("Registros en el FASTA", len(meta) + len(mal_formados)),
    ("Encabezados mal formados (sin un único `#` y un único `@`)", len(mal_formados)),
    ("Clasificaciones con menos de 3 niveles", len(pocos_niveles)),
    ("`pdb_id` duplicados", int(duplicados.nunique())),
    ("`pdb_id` con prefijo distinto de `PDB` + 8 dígitos (se aceptan)",
     int((~meta["pdb_id"].str.fullmatch(r"PDB\d{8}")).sum())),
    ("Caracteres fuera de A, C, G, T, N (en mayúsculas)", ", ".join(otros) or "ninguno"),
    ("Secuencias con minúsculas (se guardan sin modificar)", "ver `resultados/inconsistencias_fasta.csv`"),
    ("Secuencias con N", int((meta["n_percent"] > 0).sum())),
    ("Longitud mínima / máxima (nt)", f"{meta['length'].min()} / {meta['length'].max()}"),
    ("Secuencias de longitud cero", int((meta["length"] == 0).sum())),
    ("TEs sin especie tras `normalize_species`", len(sin_cruce)),
    ("Documentos en `tes` / `secuencias`", f"{db.tes.count_documents({})} / {db.secuencias.count_documents({})}"),
]
texto = "\n".join([
    "# Reporte de calidad del FASTA",
    "",
    "Generado por `notebooks/02_pipeline_fasta_tes.ipynb` al cargar `tes` y `secuencias` en "
    f"`{DB_NAME}`. Exploración previa: `notebooks/00_exploracion.ipynb` y `resultados/inconsistencias_fasta.csv`.",
    "",
    "| Hallazgo | Valor |",
    "| --- | --- |",
    *[f"| {h} | {v} |" for h, v in filas],
    "",
    "## Decisiones",
    "",
    "- `pdb_id`: se toma todo lo anterior a `#`, sin asumir el prefijo `PDB`.",
    "- `gc_content = (G + C) / (A + C + G + T)`: excluye N e IUPAC del denominador; "
    "`n_percent = N / length`. Ambos sobre la secuencia en mayúsculas.",
    "- `secuencias.sequence` guarda la secuencia original sin modificar (minúsculas incluidas).",
    "- `species_id` se obtiene con `src.normalizacion.normalize_species` (ver `resultados/reporte_cruce.md`); "
    "`species_name` guarda el nombre tal como viene en el FASTA.",
    "- `phylum` se copia de `especies` y puede ser `null` cuando falta en el CSV.",
    "",
])
(RESULTADOS / "reporte_calidad_fasta.md").write_text(texto, encoding="utf-8")
print(texto)

# Reporte de calidad del FASTA

Generado por `notebooks/02_pipeline_fasta_tes.ipynb` al cargar `tes` y `secuencias` en `panteon`. Exploración previa: `notebooks/00_exploracion.ipynb` y `resultados/inconsistencias_fasta.csv`.

| Hallazgo | Valor |
| --- | --- |
| Registros en el FASTA | 239946 |
| Encabezados mal formados (sin un único `#` y un único `@`) | 0 |
| Clasificaciones con menos de 3 niveles | 0 |
| `pdb_id` duplicados | 0 |
| `pdb_id` con prefijo distinto de `PDB` + 8 dígitos (se aceptan) | 31600 |
| Caracteres fuera de A, C, G, T, N (en mayúsculas) | ninguno |
| Secuencias con minúsculas (se guardan sin modificar) | ver `resultados/inconsistencias_fasta.csv` |
| Secuencias con N | 54837 |
| Longitud mínima / máxima (nt) | 81 / 19926 |
| Secuencias de longitud cero | 0 |
| TEs sin especie tras `normalize_species` | 0 |
| Documentos en `tes` / `secuencias` | 239946 / 239946 |

## Decisiones

- `pdb_id`: se toma todo lo anterior a `#`, sin asumir el prefijo `PDB`.
- `gc_content